# Minecraft Server Hosting Setup

This notebook sets up and runs a Minecraft server on Google Colab with SSH tunneling.

## Prerequisites
- Google Drive access
- SSH key file (qasw1234.first.pem)
- Portmap.io account for SSH tunneling

**Note**: Run cells in order. One-time setup cells are marked and only need to be run once.

## One-Time Setup

Run these cells only once to set up the environment.

In [ ]:
# Clean up existing Java installations
!sudo apt purge openjdk-* -y
!sudo apt autoremove -y

In [ ]:
# Mount Google Drive
#1 click the folder icon and select "Mount Drive"

# from google.colab import drive
# drive.mount('/content/drive')


#create server directory
!mkdir -p /content/drive/MyDrive/ms
%cd /content/drive/MyDrive/ms


In [ ]:
# Download Minecraft server (version 1.21.3)
!wget -O server.jar https://piston-data.mojang.com/v1/objects/6e64dcabba3c01a7271b4fa6bd898483b794c59b/server.jar


## Regular Server Setup

Run these cells each time you want to start the server.


In [1]:
# Install OpenJDK 21
!sudo apt update
!sudo apt install openjdk-21-jdk -y

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,747 kB]
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,050 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1

In [2]:
# Verify Java installation
!java -version

openjdk version "21.0.7" 2025-04-15
OpenJDK Runtime Environment (build 21.0.7+6-Ubuntu-0ubuntu122.04)
OpenJDK 64-Bit Server VM (build 21.0.7+6-Ubuntu-0ubuntu122.04, mixed mode, sharing)


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Verify directory contents and set SSH key permissions
%cd /content/drive/MyDrive/ms

In [12]:
!ls -l
!chmod 600 qasw1234.first.pem

total 69100
-rw-------  1 root root        2 Jun 24 06:30 banned-ips.json
-rw-------  1 root root        2 Jun 24 06:30 banned-players.json
drwx------  2 root root     4096 Jun 24 05:45 debug
-rw-------  1 root root       10 Jun 24 01:41 eula.txt
drwx------  2 root root     4096 Apr 28 06:31 frp_0.62.1_linux_amd64
-rw-------  1 root root 13170768 Apr 28 06:32 frp_0.62.1_linux_amd64.tar.gz
-rw-------  1 root root      139 Jun 24 03:13 frpc.ini
drwx------  8 root root     4096 Jun 24 01:53 libraries
drwx------  2 root root     4096 Jun 24 06:32 logs
-rw-------  1 root root      134 Jun 24 06:30 ops.json
-rw-------  1 root root     1704 Jun 23 19:49 qasw1234.first.pem
-rw-------  1 root root 57554576 Jun 17 11:21 server.jar
-rw-------  1 root root     1395 Jun 24 06:30 server.properties
-rw-------  1 root root      105 Jun 24 06:30 usercache.json
drwx------  3 root root     4096 Jun 24 01:53 versions
-rw-------  1 root root        2 Jun 24 01:53 whitelist.json
drwx------ 12 root root     

## Server Configuration and Execution

The following cell runs the Minecraft server with SSH tunneling.

**Connection Details**:
- External IP: `tcp://qasw1234-61819.portmap.io:61819`
- Maps to: `localhost:25565`

**Instructions**:
- Run the cell below to start the server
- Type Minecraft commands in the input prompt
- Type 'exit' to stop the server
- Monitor [MC] for server output and [SSH] for tunnel status

In [ ]:
import subprocess
import threading
import os
import time

# Server configuration
MIN_RAM = "6G"  # -Xms (initial RAM)
MAX_RAM = "8G"  # -Xmx (maximum RAM)
SERVER_JAR = "server.jar"

# SSH reverse tunnel command
SSH_COMMAND = [
    "ssh", "-v",
    "-i", "qasw1234.first.pem",
    "-o", "StrictHostKeyChecking=no",
    "-N",  # no remote command execution
    "-R", "61819:localhost:25565",
    "qasw1234.first@qasw1234-61819.portmap.io"
]

def cleanup_world_lock():
    """Remove any existing session.lock file"""
    lock_file = "./world/session.lock"
    if os.path.exists(lock_file):
        print("Found existing session.lock - removing it...")
        os.remove(lock_file)

def run_server():
    """Runs the Minecraft server in a subprocess"""
    cleanup_world_lock()

    command = f"java -Xms{MIN_RAM} -Xmx{MAX_RAM} -jar {SERVER_JAR} nogui"
    process = subprocess.Popen(
        command.split(),
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True,
        bufsize=1
    )
    return process

def run_ssh_tunnel():
    """Starts the SSH reverse tunnel"""
    try:
        print("Starting SSH reverse tunnel...")
        ssh_process = subprocess.Popen(
            SSH_COMMAND,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            universal_newlines=True,
            bufsize=1
        )
        print(f"SSH tunnel PID: {ssh_process.pid}")
        return ssh_process
    except Exception as e:
        print("Failed to start SSH tunnel:", e)
        return None

def output_reader(process, prefix=""):
    """Thread that continuously reads a process's output and prints it with a prefix"""
    while True:
        output = process.stdout.readline()
        if output == '' and process.poll() is not None:
            break
        if output:
            print(f"{prefix}{output.strip()}")

def server_manager():
    """Main server control function"""
    print("Starting Minecraft server... (Type 'exit' to stop)")

    process = run_server()
    if process.poll() is not None:
        print("Server failed to start!")
        return

    # Start Minecraft server output reader thread
    threading.Thread(target=output_reader, args=(process, "[MC] "), daemon=True).start()

    # Start SSH tunnel and its output reader thread
    ssh_process = run_ssh_tunnel()
    if ssh_process:
        threading.Thread(target=output_reader, args=(ssh_process, "[SSH] "), daemon=True).start()

    try:
        while True:
            if process.poll() is not None:
                print("Minecraft server has stopped unexpectedly!")
                break

            cmd = input()
            if cmd.lower() == 'exit':
                process.stdin.write("stop\n")
                process.stdin.flush()
                time.sleep(5)
                break

            if process.poll() is None:
                process.stdin.write(f"{cmd}\n")
                process.stdin.flush()
            else:
                print("Cannot send command - server is not running")
                break

    except (KeyboardInterrupt, EOFError):
        print("\nStopping Minecraft server...")
        if process.poll() is None:
            process.stdin.write("stop\n")
            process.stdin.flush()
            time.sleep(3)

    finally:
        if process.poll() is None:
            process.terminate()
        if ssh_process and ssh_process.poll() is None:
            ssh_process.terminate()
        print("Server and SSH tunnel have stopped.")

if __name__ == "__main__":
    # Check if server.jar exists
    if not os.path.exists(SERVER_JAR):
        print(f"Error: {SERVER_JAR} not found!")
        print("Please download it first using:")
        print("!wget https://piston-data.mojang.com/v1/objects/6e64dcabba3c01a7271b4fa6bd898483b794c59b/server.jar -O server.jar")
    else:
        server_manager()


Starting Minecraft server... (Type 'exit' to stop)
Found existing session.lock - removing it...
Starting SSH reverse tunnel...
SSH tunnel PID: 15435
[SSH] OpenSSH_8.9p1 Ubuntu-3ubuntu0.13, OpenSSL 3.0.2 15 Mar 2022
[SSH] debug1: Reading configuration data /etc/ssh/ssh_config
[SSH] debug1: /etc/ssh/ssh_config line 19: include /etc/ssh/ssh_config.d/*.conf matched no files
[SSH] debug1: /etc/ssh/ssh_config line 21: Applying options for *
[SSH] debug1: Connecting to qasw1234-61819.portmap.io [193.161.193.99] port 22.
[SSH] debug1: Connection established.
[SSH] debug1: identity file qasw1234.first.pem type -1
[SSH] debug1: identity file qasw1234.first.pem-cert type -1
[SSH] debug1: Local version string SSH-2.0-OpenSSH_8.9p1 Ubuntu-3ubuntu0.13
[SSH] debug1: Remote protocol version 2.0, remote software version OpenSSH_7.2p2 Ubuntu-4ubuntu2.10
[SSH] debug1: compat_banner: match: OpenSSH_7.2p2 Ubuntu-4ubuntu2.10 pat OpenSSH_7.0*,OpenSSH_7.1*,OpenSSH_7.2*,OpenSSH_7.3*,OpenSSH_7.5*,OpenSSH_7.6*,O